# EXP_009d1: Attractor Dominance, Basin Mapping & Pathway Analysis

## Depends On
**Stage 0 (Reproducibility Gate) must PASS before running this notebook.**

## Hypotheses Under Test

### H1: Attractor Dominance
The `prolet` attractor is the dominant basin of GPT-2 Small's weight geometry. It should capture the majority of a diverse prompt set, regardless of input register, topic, or complexity.

### H2: Secondary Basin Existence
The `Divine` attractor is a genuine secondary basin, not a one-off artefact of a single prompt. Other prompts with similar syntactic properties should also route to `Divine` (or to other previously unseen basins).

### H3: Dissolution Pathway Structure
The intermediate tokens observed during dissolution (e.g., `Femminus Fem`) reflect the statistical topology of the training corpus, not random noise. Different input types may trace different but internally coherent pathways to the same terminal attractor.

---


In [ ]:
# ============================================================
# STEP 0: DEPENDENCIES
# ============================================================
import sys
!{sys.executable} -m pip install kaleido -q

In [ ]:
# ============================================================
# STEP 1: CALIBRATION
# ============================================================
import torch
import numpy as np
import os
import plotly.graph_objects as go
import plotly.express as px
from transformer_lens import HookedTransformer
from IPython.display import Markdown, display
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2-small", device=device)
print(f"Architecture: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads, d_model={model.cfg.d_model}")
print(f"Running on: {device}")

# Output directory for all saved artifacts
OUTPUT_DIR = "output_stage1"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")

In [ ]:
# ============================================================
# STEP 2: CONFIGURATION — 125 Prompts from prompt_library.py
# ============================================================
from prompt_library import (
    PROMPT_LIBRARY, PREDICTIONS, CATEGORY_MAP,
    COMPLEX, NARRATIVE, SIMPLE, CHEMICAL, ACRONYMS, VULGARITY, WILD
)

# Tightened schedule: convergence occurs by ~100
ITERATION_SCHEDULE = [0, 2, 3, 5, 10, 20, 50, 100]
MAX_ITERATIONS = max(ITERATION_SCHEDULE)

LAYER_START = 0
LAYER_END = model.cfg.n_layers - 1

print(f"Schedule: {ITERATION_SCHEDULE}")
print(f"Room: Layers {LAYER_START} → {LAYER_END}")
print(f"Total prompts: {len(PROMPT_LIBRARY)}")
print(f"\nBreakdown:")
for cat_name, cat_dict in [
    ("Complex", COMPLEX), ("Narrative", NARRATIVE),
    ("Simple", SIMPLE), ("Chemical", CHEMICAL),
    ("Acronyms", ACRONYMS), ("Vulgarity", VULGARITY),
    ("Wild", WILD)
]:
    print(f"  {cat_name}: {len(cat_dict)} prompts")

# Save config
config_md = f"""# Stage 1 Run Config\n
- **Prompts:** {len(PROMPT_LIBRARY)}\n
- **Schedule:** {ITERATION_SCHEDULE}\n
- **Layers:** {LAYER_START} → {LAYER_END}\n
- **Device:** {device}\n
"""
with open(os.path.join(OUTPUT_DIR, 'config.md'), 'w') as f:
    f.write(config_md)
print(f"\n[SAVED] {OUTPUT_DIR}/config.md")

In [ ]:
# ============================================================
# STEP 3: THE CORE ENGINE — Identical to lucier_total_resonance
# ============================================================

def get_top_tokens(model, resid_vector, k=5):
    """Decode a residual stream vector into top-k token predictions.
    Applies the Final LayerNorm before unembedding for correct decoding."""
    normalized = model.ln_final(resid_vector)
    logits = normalized @ model.W_U + model.b_U
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_indices = torch.topk(probs, k)
    tokens = [model.tokenizer.decode([idx]) for idx in top_indices]
    return list(zip(tokens, top_probs.tolist()))


def run_total_resonance_loop(model, prompt, layer_start, layer_end, max_iter, schedule):
    """
    TOTAL Lucier Loop: iteratively re-inject the ENTIRE residual stream
    tensor (all token positions) through the layer slice.
    Returns a list of snapshot dicts at each scheduled iteration.
    """
    snapshots = []
    hook_point_read = f"blocks.{layer_end}.hook_resid_post"
    hook_point_write = f"blocks.{layer_start}.hook_resid_pre"
    
    with torch.no_grad():
        _, cache = model.run_with_cache(
            prompt,
            names_filter=lambda n: n == hook_point_read
        )
    
    current_tensor = cache[hook_point_read][0].clone()
    seq_len = current_tensor.shape[0]
    initial_norm = current_tensor.norm().item()
    
    last_vec = current_tensor[-1, :].clone()
    mean_vec = current_tensor.mean(dim=0).clone()
    
    if 0 in schedule:
        top_tokens_last = get_top_tokens(model, last_vec)
        all_pos_tokens = []
        for pos in range(seq_len):
            pos_top = get_top_tokens(model, current_tensor[pos, :], k=1)
            all_pos_tokens.append(pos_top[0][0])
        snapshots.append({
            "iteration": 0,
            "tensor": current_tensor.clone().cpu(),
            "last_vector": last_vec.clone().cpu(),
            "mean_vector": mean_vec.clone().cpu(),
            "last_norm": last_vec.norm().item(),
            "mean_norm": mean_vec.norm().item(),
            "tensor_norm": current_tensor.norm().item(),
            "top_tokens": top_tokens_last,
            "all_position_tokens": all_pos_tokens,
            "cosine_sim_last": 1.0,
            "cosine_sim_mean": 1.0,
            "position_similarity": 1.0,
        })
    
    prev_last = last_vec.clone()
    prev_mean = mean_vec.clone()
    
    for i in range(1, max_iter + 1):
        # Normalise to maintain energy level
        current_norm = current_tensor.norm().item()
        if current_norm > 0:
            current_tensor = current_tensor * (initial_norm / current_norm)
        
        inject_tensor = current_tensor.clone()
        
        def injection_hook(resid, hook, tensor=inject_tensor):
            resid[0, :, :] = tensor
            return resid
        
        model.add_hook(hook_point_write, injection_hook)
        try:
            with torch.no_grad():
                _, cache = model.run_with_cache(
                    prompt,
                    names_filter=lambda n: n == hook_point_read
                )
        finally:
            model.reset_hooks()
        
        current_tensor = cache[hook_point_read][0].clone()
        last_vec = current_tensor[-1, :].clone()
        mean_vec = current_tensor.mean(dim=0).clone()
        
        if i in schedule:
            cos_sim_last = torch.nn.functional.cosine_similarity(
                last_vec.unsqueeze(0), prev_last.unsqueeze(0)
            ).item()
            cos_sim_mean = torch.nn.functional.cosine_similarity(
                mean_vec.unsqueeze(0), prev_mean.unsqueeze(0)
            ).item()
            
            pos_norms = current_tensor.norm(dim=1, keepdim=True).clamp(min=1e-8)
            normalized_positions = current_tensor / pos_norms
            pos_sim_matrix = normalized_positions @ normalized_positions.T
            mask = ~torch.eye(seq_len, dtype=torch.bool, device=pos_sim_matrix.device)
            position_similarity = pos_sim_matrix[mask].mean().item()
            
            top_tokens_last = get_top_tokens(model, last_vec)
            all_pos_tokens = []
            for pos in range(seq_len):
                pos_top = get_top_tokens(model, current_tensor[pos, :], k=1)
                all_pos_tokens.append(pos_top[0][0])
            
            snapshots.append({
                "iteration": i,
                "tensor": current_tensor.clone().cpu(),
                "last_vector": last_vec.clone().cpu(),
                "mean_vector": mean_vec.clone().cpu(),
                "last_norm": last_vec.norm().item(),
                "mean_norm": mean_vec.norm().item(),
                "tensor_norm": current_tensor.norm().item(),
                "top_tokens": top_tokens_last,
                "all_position_tokens": all_pos_tokens,
                "cosine_sim_last": cos_sim_last,
                "cosine_sim_mean": cos_sim_mean,
                "position_similarity": position_similarity,
            })
            print(f"  iter {i:>3}: top='{top_tokens_last[0][0].strip()}', "
                  f"cos_mean={cos_sim_mean:.4f}, pos_collapse={position_similarity:.4f}")
        
        prev_last = last_vec.clone()
        prev_mean = mean_vec.clone()
    
    return snapshots

print("Engine loaded.")

In [ ]:
# ============================================================
# STEP 4: RUN ALL 125 PROMPTS
# ============================================================

all_results = {}

for idx, (label, prompt) in enumerate(PROMPT_LIBRARY.items()):
    print(f"\n{'='*60}")
    print(f"[{idx+1}/{len(PROMPT_LIBRARY)}] RECORDING: '{label}'")
    print(f"  Prompt: \"{prompt}\"")
    print(f"{'='*60}")
    
    snapshots = run_total_resonance_loop(
        model, prompt,
        layer_start=LAYER_START,
        layer_end=LAYER_END,
        max_iter=MAX_ITERATIONS,
        schedule=ITERATION_SCHEDULE
    )
    all_results[label] = snapshots
    
    terminal = snapshots[-1]['top_tokens'][0][0].strip()
    print(f"  ✓ Terminal token: '{terminal}'")

print(f"\n{'='*60}")
print(f"ALL {len(all_results)} RECORDINGS COMPLETE.")

---
## 5. Analysis

### 5a. Hypothesis Assessment — Predictions vs Actuals

In [ ]:
# ============================================================
# VIS 5a: HYPOTHESIS ASSESSMENT — Predictions vs Actuals
# ============================================================

md = "# Stage 1 Results: Hypothesis Assessment\n\n"
md += "| Prompt | Category | Predicted | Actual Terminal | Match? |\n"
md += "|:---|:---|:---|:---|:---|\n"

basin_counts = {}
category_basins = {}
mismatches = []

for label, snapshots in all_results.items():
    terminal = snapshots[-1]['top_tokens'][0][0].strip()
    predicted_basin, confidence = PREDICTIONS[label]
    category = CATEGORY_MAP[label]
    
    # Classify actual basin
    if 'prolet' in terminal or terminal in 'prolet':
        actual_basin = 'prolet'
    elif 'Divine' in terminal or terminal in 'Divine':
        actual_basin = 'Divine'
    else:
        actual_basin = f'OTHER:{terminal}'
    
    basin_counts[actual_basin] = basin_counts.get(actual_basin, 0) + 1
    
    if category not in category_basins:
        category_basins[category] = []
    category_basins[category].append((label, actual_basin, terminal))
    
    match = '✓' if predicted_basin.lower() in actual_basin.lower() else '✗'
    if match == '✗' and predicted_basin != 'unknown':
        mismatches.append((label, predicted_basin, actual_basin))
    
    md += f"| {label} | {category} | `{predicted_basin}` ({confidence}) | `{terminal}` → **{actual_basin}** | {match} |\n"

md += "\n---\n\n"
md += "## Basin Summary\n\n"
md += "| Basin | Count | % |\n"
md += "|:---|:---|:---|\n"
total = len(all_results)
for basin, count in sorted(basin_counts.items(), key=lambda x: -x[1]):
    md += f"| **{basin}** | {count} | {count/total*100:.1f}% |\n"

md += "\n---\n\n"
md += "## Category Breakdown\n\n"
for cat, entries in category_basins.items():
    md += f"### {cat} ({len(entries)} prompts)\n"
    cat_basins = {}
    for label, basin, tok in entries:
        cat_basins[basin] = cat_basins.get(basin, 0) + 1
    for b, c in sorted(cat_basins.items(), key=lambda x: -x[1]):
        md += f"- {b}: {c}/{len(entries)}\n"
    md += "\n"

if mismatches:
    md += "## Prediction Mismatches\n\n"
    for label, pred, actual in mismatches:
        md += f"- **{label}**: predicted `{pred}`, got `{actual}`\n"

# Save and display
with open(os.path.join(OUTPUT_DIR, 'hypothesis_assessment.md'), 'w') as f:
    f.write(md)
print(f"[SAVED] {OUTPUT_DIR}/hypothesis_assessment.md")
display(Markdown(md))

### 5b. Cross-Prompt Convergence Matrix

In [ ]:
# ============================================================
# VIS 5b: CROSS-PROMPT CONVERGENCE MATRIX
# ============================================================

labels = list(all_results.keys())
n = len(labels)
sim_matrix = np.zeros((n, n))

final_vectors = []
for label in labels:
    final_vec = all_results[label][-1]["mean_vector"]
    final_vectors.append(final_vec)

for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = torch.nn.functional.cosine_similarity(
            final_vectors[i].unsqueeze(0).float(),
            final_vectors[j].unsqueeze(0).float()
        ).item()

fig_sim = px.imshow(
    sim_matrix,
    x=labels, y=labels,
    color_continuous_scale="Viridis",
    title="Stage 1: Cross-Prompt Convergence (125 Prompts)",
    aspect="auto",
)
fig_sim.update_layout(template="plotly_dark", height=900, width=1200)
fig_sim.show()
fig_sim.write_image(os.path.join(OUTPUT_DIR, 'convergence_matrix.png'), scale=2)
print(f"[SAVED] {OUTPUT_DIR}/convergence_matrix.png")

off_diag = sim_matrix[np.triu_indices(n, k=1)]
print(f"\nMean cross-prompt similarity: {off_diag.mean():.4f}")
print(f"Min:  {off_diag.min():.4f}")
print(f"Max:  {off_diag.max():.4f}")

### 5c. Dissolution Pathway Analysis

In [ ]:
# ============================================================
# VIS 5c: DISSOLUTION PATHWAYS — Per Category
# ============================================================

md = "# Dissolution Pathways — Last-Token Top Prediction\n\n"

for cat_name, cat_dict in [
    ("Complex", COMPLEX), ("Narrative", NARRATIVE),
    ("Simple", SIMPLE), ("Chemical", CHEMICAL),
    ("Acronyms", ACRONYMS), ("Vulgarity", VULGARITY),
    ("Wild", WILD)
]:
    cat_labels = [k for k in cat_dict.keys() if k in all_results]
    if not cat_labels:
        continue
    
    md += f"## {cat_name} ({len(cat_labels)} prompts)\n\n"
    md += "| Iter | " + " | ".join(cat_labels[:10]) + " |\n"
    md += "| :--- | " + " | ".join([":---"] * min(len(cat_labels), 10)) + " |\n"
    
    for idx, iteration in enumerate(ITERATION_SCHEDULE):
        row = f"| **{iteration}** |"
        for label in cat_labels[:10]:
            snapshots = all_results[label]
            if idx < len(snapshots):
                tok = snapshots[idx]['top_tokens'][0][0]
                clean_t = tok.replace('\n', '↵').replace('`', "'").strip()
                row += f" `{clean_t}` |"
            else:
                row += " — |"
        md += row + "\n"
    md += "\n"

with open(os.path.join(OUTPUT_DIR, 'dissolution_pathways.md'), 'w') as f:
    f.write(md)
print(f"[SAVED] {OUTPUT_DIR}/dissolution_pathways.md")
display(Markdown(md))

### 5d. Sentence Dissolution Tables — Full Position Reconstruction

In [ ]:
# ============================================================
# VIS 5d: SENTENCE DISSOLUTION TABLES
# ============================================================

md = "# Full Sentence Dissolution — All 125 Prompts\n\n"

for label in PROMPT_LIBRARY.keys():
    if label not in all_results:
        continue
    snapshots = all_results[label]
    predicted, conf = PREDICTIONS[label]
    category = CATEGORY_MAP[label]
    terminal = snapshots[-1]['top_tokens'][0][0].strip()
    
    md += f"### {label} [{category}] → `{terminal}` (predicted: `{predicted}`)\n"
    md += f"*\"{PROMPT_LIBRARY[label]}\"*\n\n"
    md += "| Iter | Reconstructed Output |\n"
    md += "|:---|:---|\n"
    for s in snapshots:
        tokens = s['all_position_tokens']
        clean = [t.replace('\n', '↵').replace('|', '\\|') for t in tokens]
        sentence = ' '.join(clean)
        md += f"| {s['iteration']} | {sentence} |\n"
    md += "\n"

with open(os.path.join(OUTPUT_DIR, 'dissolution_sentences.md'), 'w') as f:
    f.write(md)
print(f"[SAVED] {OUTPUT_DIR}/dissolution_sentences.md")
print(f"Full dissolution tables: {len(all_results)} prompts written.")

### 5e. 3D PCA Trajectories — All Prompts

In [ ]:
# ============================================================
# VIS 5e: 3D PCA TRAJECTORIES
# ============================================================
from sklearn.decomposition import PCA
import pandas as pd

all_vecs = []
labels_list = []
cats_list = []
iters_list = []
text_list = []

for label, snapshots in all_results.items():
    for s in snapshots:
        all_vecs.append(s["mean_vector"].detach().cpu().numpy())
        labels_list.append(label)
        cats_list.append(CATEGORY_MAP.get(label, 'Unknown'))
        iters_list.append(s["iteration"])
        top_tok = s['top_tokens'][0][0].replace('\n', '↵').strip()
        text_list.append(f"Iter {s['iteration']}: {top_tok}")

all_vecs = np.array(all_vecs)
pca = PCA(n_components=3)
vecs_3d = pca.fit_transform(all_vecs)

df = pd.DataFrame({
    'x': vecs_3d[:, 0],
    'y': vecs_3d[:, 1],
    'z': vecs_3d[:, 2],
    'Prompt': labels_list,
    'Category': cats_list,
    'Iteration': iters_list,
    'Top_Token': text_list
})

# Color by category for readability
fig_topo = px.line_3d(
    df, x='x', y='y', z='z',
    color='Category',
    hover_name='Top_Token',
    markers=True,
    title=f"Stage 1: Attractor Landscape — {len(all_results)} Prompt Trajectories<br>"
          f"<sup>(Explained Variance: {sum(pca.explained_variance_ratio_)*100:.1f}%)</sup>"
)
fig_topo.update_traces(marker=dict(size=3), line=dict(width=2))
fig_topo.update_layout(
    template="plotly_dark",
    height=900,
    width=1200,
    scene=dict(
        xaxis_title="PC 1",
        yaxis_title="PC 2",
        zaxis_title="PC 3",
    )
)
fig_topo.show()
fig_topo.write_image(os.path.join(OUTPUT_DIR, 'topology_3d.png'), scale=2)
print(f"[SAVED] {OUTPUT_DIR}/topology_3d.png")

### 5f. Basin Distribution Chart

In [ ]:
# ============================================================
# VIS 5f: BASIN DISTRIBUTION BAR CHART
# ============================================================

basin_data = []
for label, snapshots in all_results.items():
    terminal = snapshots[-1]['top_tokens'][0][0].strip()
    category = CATEGORY_MAP.get(label, 'Unknown')
    if 'prolet' in terminal or terminal in 'prolet':
        basin = 'prolet'
    elif 'Divine' in terminal or terminal in 'Divine':
        basin = 'Divine'
    else:
        basin = terminal
    basin_data.append({'Prompt': label, 'Category': category, 'Basin': basin})

basin_df = pd.DataFrame(basin_data)
basin_summary = basin_df.groupby(['Category', 'Basin']).size().reset_index(name='Count')

fig_basin = px.bar(
    basin_summary, x='Category', y='Count', color='Basin',
    title=f"Stage 1: Basin Distribution by Category ({len(all_results)} prompts)",
    barmode='stack'
)
fig_basin.update_layout(template="plotly_dark", height=500, width=900)
fig_basin.show()
fig_basin.write_image(os.path.join(OUTPUT_DIR, 'basin_distribution.png'), scale=2)
print(f"[SAVED] {OUTPUT_DIR}/basin_distribution.png")

In [ ]:
# ============================================================
# STEP 6: SAVE RAW DATA
# ============================================================

save_data = {}
for label, snapshots in all_results.items():
    save_data[label] = {
        "iterations": [s["iteration"] for s in snapshots],
        "last_vectors": torch.stack([s["last_vector"] for s in snapshots]),
        "mean_vectors": torch.stack([s["mean_vector"] for s in snapshots]),
        "last_norms": [s["last_norm"] for s in snapshots],
        "mean_norms": [s["mean_norm"] for s in snapshots],
        "cosine_sims_last": [s["cosine_sim_last"] for s in snapshots],
        "cosine_sims_mean": [s["cosine_sim_mean"] for s in snapshots],
        "position_similarity": [s["position_similarity"] for s in snapshots],
        "top_tokens": [s["top_tokens"] for s in snapshots],
        "all_position_tokens": [s["all_position_tokens"] for s in snapshots],
    }

torch.save(save_data, os.path.join(OUTPUT_DIR, 'stage1_results.pt'))
print(f"[SAVED] {OUTPUT_DIR}/stage1_results.pt")

config = {
    "schedule": ITERATION_SCHEDULE,
    "layer_start": LAYER_START,
    "layer_end": LAYER_END,
    "prompt_count": len(PROMPT_LIBRARY),
    "model": "gpt2-small",
    "mode": "stage1_attractor_dominance",
}
torch.save(config, os.path.join(OUTPUT_DIR, 'stage1_config.pt'))
print(f"[SAVED] {OUTPUT_DIR}/stage1_config.pt")
print(f"\n✓ All artifacts saved to {os.path.abspath(OUTPUT_DIR)}")